# Gravity observations, reference and sign

**Level 1 · Prerequisites: none; reading time: 25 minutes.**

## Objectives

Establish the measurand before applying a correction: distinguish absolute gravity, normal gravity and Bouguer disturbance; preserve upward-positive height and downward-positive gravity explicitly.

## Setup

Use the repository's reviewed Python 3.12 notebook environment. This notebook installs nothing. Inspect the station’s committed analytic-control metadata: WGS84 ellipsoidal height, mGal gravity, tide-free convention and synthetic provenance. This is a dimensional/sign exercise, not a field observation.

The next cell reads the serialized station and reports its physical convention.

In [ ]:
import json
from pathlib import Path

# Inspect the actual product control and its declared physical metadata.
root = Path.cwd().parents[2]
path = root / 'docs/methods/gravity-processing/examples/station-control.json'
request = json.loads(path.read_text(encoding='utf-8'))
dataset = request['dataset']
station = dataset['stations'][0]
meta = dataset['metadata']
print({'state': dataset['state'], 'source_kind': meta['source_kind'], 'crs': meta['crs'],
       'height_datum': meta['height_datum'], 'height_sign': meta['height_sign'],
       'gravity_unit': meta['gravity_unit'], 'gravity_sign': meta['gravity_sign'],
       'latitude_deg': station['latitude_deg'], 'height_m': station['receiver_height_m'],
       'absolute_gravity_mgal': station['value_mgal']})

## Data

Inspect the station’s committed analytic-control metadata: WGS84 ellipsoidal height, mGal gravity, tide-free convention and synthetic provenance. This is a dimensional/sign exercise, not a field observation.

**File:** `docs/methods/gravity-processing/examples/station-control.json`.

## Method

The conventional sequence forms a reference-dependent anomaly before terrain or regional interpretation. For an ideal infinite slab, Δg=2πGρt; converting SI acceleration to mGal multiplies by 10⁵. Here G [m³ kg⁻¹ s⁻²], density ρ [kg m⁻³], and thickness t [m] yield m s⁻² before conversion. A station height is not slab thickness; terrain and datum choices remain explicit. Harmonica documents the slab-attraction calculation ([API reference](https://www.fatiando.org/harmonica/latest/api/index.html)); field procedures and QA context are summarized by the USGS ([Techniques and Methods 2-D4](https://pubs.usgs.gov/tm/02/d04/tm2d4.pdf)).

The processor requires the reference chain to be explicit. It first converts the recorded gravity sign/unit to downward-positive mGal, evaluates normal gravity γ(φ,h) on the declared ellipsoid, then applies elevation and plate terms. The equivalent decomposition is D_B=[g−γ(φ,0)]+[γ(φ,0)−γ(φ,h)]−B. The second bracket is the free-air elevation contribution under this normal-field convention; surface slab thickness is separate. For a land station above datum, a positive slab attraction is subtracted. A different terrain correction requires terrain evidence and its own signed residual; the infinite slab is not a DEM correction.

The pipeline also records calibration, instrument drift, tide status, CRS, height datum and lineage. Ellipsoidal and orthometric heights differ by geoid undulation; silently mixing them biases the normal-gravity and elevation terms. The plot calls the implemented processor and shows the signed terms and the cumulative disturbance for the actual one-station analytic control. It is an audit of bookkeeping, not a claim of field accuracy or a spatial gravity map.

## Results

The artifact identifies a single analytic station at 45° and 1000 m; source provenance labels it synthetic. Metadata prevents treating its numeric gravity as an unqualified field value.

The processor’s history supplies the stage increments; compute the first reference anomaly from its recorded first increment and use the later recorded additions. The comparison of uncertainty policies is conditional on the same input sigmas.


In [ ]:
import copy, json, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Execute the real M01 pipeline for the committed station control under both declared uncertainty policies.
root = Path.cwd().parents[2]
sys.path.insert(0, str(root / 'data-pipeline'))
from gravity_processing import process_survey
q = json.loads((root / 'docs/methods/gravity-processing/examples/station-control.json').read_text(encoding='utf-8'))
base = q['dataset']; conservative = process_survey(base, q['config'])
config = copy.deepcopy(q['config']); config['uncertainty_model'] = 'independent_first_order'
independent = process_survey(base, config)
h = conservative['dataset']['history']; g = base['stations'][0]['value_mgal']
terms = [g + h[0]['additions_mgal'][0], h[1]['additions_mgal'][0], h[2]['additions_mgal'][0]]
cumulative = np.cumsum(terms)
assert np.isclose(cumulative[-1], conservative['dataset']['stations'][0]['value_mgal'], atol=1e-9, rtol=0)
fig,(ax,bx)=plt.subplots(2,1,figsize=(9,7),constrained_layout=True)
labels=['g−γ(φ,0)','γ(φ,0)−γ(φ,h)','−B plate']
ax.bar(labels,terms,color=['#4477aa','#66ccee','#cc6677']); ax.plot(range(3),cumulative,'ko--',label='Cumulative disturbance')
ax.axhline(0,color='.4',lw=.8); ax.set(ylabel='Contribution [mGal]',title='One synthetic station: signed correction chain'); ax.legend()
bx.bar(['Independent RSS','Conservative marginals'],[independent['processing']['uncertainty_mgal'][0],conservative['processing']['uncertainty_mgal'][0]],color=['#228833','#aa3377'])
bx.set(ylabel='Propagated σ [mGal]',title='Two declared aggregation policies; conditional uncertainty')
print('derived mGal:',cumulative[-1]); print('components:',conservative['processing']['uncertainty_components_mgal']); print('full method accepted:',conservative['processing']['full_method_accepted'])
plt.show()


## Summary

Never infer reference, units, sign, datum or terrain treatment from a number alone.


## References

- https://pubs.usgs.gov/tm/02/d04/tm2d4.pdf
- https://www.fatiando.org/harmonica/latest/api/index.html